# N-gram Language Model
### Next-Word Prediction using Alice's Adventures in Wonderland

**Corpus:** Alice's Adventures in Wonderland — Lewis Carroll  
**Source:** Project Gutenberg, eBook #11

## Objective

Build a statistical language model that predicts and generates text using N-grams.

This project covers:
1. N-gram counts converted into probabilities.
2. Bigram and trigram text generation.
3. Add-k smoothing for unseen N-grams.
4. Perplexity evaluation on held-out test data.
5. Comparison of model performance across configurations.

In [17]:
import re
import random
import math
from collections import defaultdict, Counter

random.seed(42)

# Load the downloaded book
with open("alice.txt", "r", encoding="utf-8") as file:
    text = file.read()

# Remove Project Gutenberg header and footer
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start = text.find(start_marker)
end = text.find(end_marker)

if start != -1 and end != -1:
    text = text[start:].split("\n", 1)[1]
    text = text[:text.find(end_marker)]

text = text.strip()

# Tokenize
tokens = re.findall(r"\w+|[^\w\s]", text.lower())

print("Cleaned characters:", len(text))
print("Total tokens:", len(tokens))
print("Preview:", tokens[:30])

Cleaned characters: 144599
Total tokens: 35519
Preview: ['[', 'illustration', ']', 'alice', '’', 's', 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', 'the', 'millennium', 'fulcrum', 'edition', '3', '.', '0', 'contents', 'chapter', 'i', '.', 'down', 'the', 'rabbit', '-', 'hole', 'chapter', 'ii']


## 1. Dataset Preparation and N-gram Counts

The corpus is divided into:
- 80% training data
- 20% testing data

An N-gram is a sequence of N consecutive tokens:
- Unigram: one token
- Bigram: two consecutive tokens
- Trigram: three consecutive tokens

We use `<s>` for the beginning of a sequence and `</s>` for its end.

In [18]:
# Train-test split
split_index = int(len(tokens) * 0.8)

train_tokens = tokens[:split_index]
test_tokens = tokens[split_index:]

def build_ngrams(tokens, n):
    counts = defaultdict(Counter)
    padded_tokens = ["<s>"] * (n - 1) + tokens + ["</s>"]

    for i in range(n - 1, len(padded_tokens)):
        context = tuple(padded_tokens[i - n + 1:i])
        next_word = padded_tokens[i]
        counts[context][next_word] += 1

    return counts

bigram_counts = build_ngrams(train_tokens, 2)
trigram_counts = build_ngrams(train_tokens, 3)

print("Training tokens:", len(train_tokens))
print("Testing tokens:", len(test_tokens))
print("Bigram contexts:", len(bigram_counts))
print("Trigram contexts:", len(trigram_counts))

Training tokens: 28415
Testing tokens: 7104
Bigram contexts: 2392
Trigram contexts: 12378


## 2. Counts to Probabilities

Maximum Likelihood Estimation (MLE) calculates the probability of a word given its context.

\[
P(w|c) = \frac{Count(c,w)}{Count(c)}
\]

The probability of a next word is its frequency divided by the total frequency of all words following that context.

In [19]:
def get_probabilities(counts):
    probabilities = {}

    for context, next_words in counts.items():
        total = sum(next_words.values())

        probabilities[context] = {
            word: count / total
            for word, count in next_words.items()
        }

    return probabilities


bigram_probs = get_probabilities(bigram_counts)
trigram_probs = get_probabilities(trigram_counts)


def suggest_next(context, probabilities, top_k=5):
    context = tuple(context)

    if context not in probabilities:
        return []

    return sorted(
        probabilities[context].items(),
        key=lambda item: item[1],
        reverse=True
    )[:top_k]


print("Next-word suggestions after 'alice':")
print(suggest_next(["alice"], bigram_probs))

Next-word suggestions after 'alice':
[(',', 0.2088235294117647), ('.', 0.11470588235294117), ('was', 0.04411764705882353), (';', 0.04411764705882353), ('thought', 0.03529411764705882)]


## 3. Text Generation

The model generates text autoregressively by repeatedly predicting and sampling the next word using the previous N-1 tokens.

## 4. Add-k Smoothing

Unseen N-grams receive zero probability under basic MLE.

Add-k smoothing addresses this by adding a small constant k to every possible next-word count.

\[
P(w|c) = \frac{Count(c,w)+k}{Count(c)+k|V|}
\]

Where:
- k is the smoothing parameter.
- V is the vocabulary.
- |V| is the vocabulary size.

In [20]:
def generate_text(probabilities, n, max_words=50):
    context = ["<s>"] * (n - 1)
    generated = []

    for _ in range(max_words):
        key = tuple(context)

        if key not in probabilities:
            break

        words = list(probabilities[key].keys())
        weights = list(probabilities[key].values())

        next_word = random.choices(words, weights=weights, k=1)[0]

        if next_word == "</s>":
            break

        generated.append(next_word)

        if n > 1:
            context = context[1:] + [next_word]

    return " ".join(generated)


print("BIGRAM GENERATION:")
print(generate_text(bigram_probs, 2))

print("\nTRIGRAM GENERATION:")
print(generate_text(trigram_probs, 3))


# Vocabulary from training data
vocabulary = set(train_tokens)
vocabulary.add("</s>")
vocab_size = len(vocabulary)


def smoothed_probability(context, word, counts, k):
    context_counts = counts.get(tuple(context), Counter())
    count = context_counts.get(word, 0)
    total = sum(context_counts.values())

    return (count + k) / (total + k * vocab_size)


print("\nSmoothed probability:")
print(smoothed_probability(("alice",), "rabbit", bigram_counts, 0.1))

BIGRAM GENERATION:
[ illustration ] alice said the king ’ t much as much to herself talking to herself . however , who were placed along hand on the mock turtle went hunting about , listening , i shan ’ m a serpent . “ now , “ well , that dark

TRIGRAM GENERATION:
[ illustration ] alice ’ s hardly worth while finishing the game . chapter ix . the pool of tears “ curiouser and curiouser ! ” chapter vii . a mad tea - things are worse than ever : she found her head . still she went on : “

Smoothed probability:
0.00017265193370165745


In [22]:
# 1. Handle rare words in training data
word_counts = Counter(train_tokens)
min_frequency = 2

mapped_train = [
    word if word_counts[word] >= min_frequency else "<unk>"
    for word in train_tokens
]

# 2. Create vocabulary
vocabulary = set(mapped_train)
vocabulary.add("</s>")
vocabulary.add("<unk>")

vocab_size = len(vocabulary)

# 3. Rebuild N-gram counts
bigram_counts = build_ngrams(mapped_train, 2)
trigram_counts = build_ngrams(mapped_train, 3)


# 4. Map unseen test words to <unk>
mapped_test = [
    word if word in vocabulary else "<unk>"
    for word in test_tokens
]


# 5. Calculate perplexity using preceding training context
def calculate_perplexity(test_data, train_data, counts, n, k):
    history = train_data[-(n - 1):] if n > 1 else []
    sequence = history + test_data + ["</s>"]

    log_probability = 0
    total_words = 0

    for i in range(len(history), len(sequence)):
        context = tuple(sequence[i - n + 1:i])
        word = sequence[i]

        probability = smoothed_probability(
            context, word, counts, k
        )

        log_probability += math.log(probability)
        total_words += 1

    return math.exp(-log_probability / total_words)


# 6. Compare models and smoothing values
k_values = [0.01, 0.1, 0.5, 1.0]

print("CORRECTED PERPLEXITY COMPARISON")
print("-" * 45)

for n, counts in [(2, bigram_counts), (3, trigram_counts)]:
    print(f"\n{'Bigram' if n == 2 else 'Trigram'}:")

    for k in k_values:
        pp = calculate_perplexity(
            mapped_test, mapped_train, counts, n, k
        )
        print(f"k={k}: Perplexity={pp:.2f}")

CORRECTED PERPLEXITY COMPARISON
---------------------------------------------

Bigram:
k=0.01: Perplexity=81.31
k=0.1: Perplexity=92.93
k=0.5: Perplexity=147.72
k=1.0: Perplexity=196.51

Trigram:
k=0.01: Perplexity=224.27
k=0.1: Perplexity=319.64
k=0.5: Perplexity=513.00
k=1.0: Perplexity=631.58


## 5. Observations and Findings

### 1. Bigram vs. Trigram
The bigram model achieved lower perplexity than the trigram model for every tested smoothing value.

The best bigram perplexity was 81.31, while the best trigram perplexity was 224.27.

### 2. Effect of Smoothing
Increasing k from 0.01 to 1.0 consistently increased perplexity for both models.

Smaller smoothing values performed better on this corpus.

### 3. Effect of Context Size
Although trigrams use more context, they performed worse because their context combinations were less frequent in the training data. This demonstrates the data sparsity problem.

### 4. Effect of Training Data
A larger corpus could provide more examples of trigram contexts and potentially improve trigram predictions.

## Conclusion

This project demonstrates how statistical language models learn word-transition patterns using N-gram counts, Maximum Likelihood Estimation, Add-k smoothing, and perplexity evaluation.

The experiments showed that additional context does not always improve prediction performance, especially when training data is limited.